# Regularization Techniques — Customer Churn Clean Solution

Compares Base model vs Dropout vs Weight Decay (L2) on AtliQ churn data.

**Key concepts:** Dropout, L2 weight decay, BatchNorm, L1 regularization term, early observation of overfitting.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Revision: L1 Regularization Term

In [ ]:
weights = torch.tensor([0.5, -0.3, 0.8, -1.5], requires_grad=True)
l1_regularization = torch.sum(torch.abs(weights))
l1_regularization.backward()
print(f"L1 Regularization: {l1_regularization.item():.4f}")
print(f"Gradients: {weights.grad}")

## Revision: Manual Batch Normalization

In [ ]:
inputs = torch.tensor([[10.0, 20.0], [15.0, 25.0], [12.0, 18.0]])
mean = inputs.mean(dim=0)
std = inputs.std(dim=0, unbiased=False)
normalized = (inputs - mean) / (std + 1e-8)
print(f"Mean: {mean}\nStd: {std}\nNormalized:\n{normalized}")

## Main Task: Data Preparation

In [ ]:
data = pd.read_csv("AtliQ_Churn_Prediction_Codebasics_DL.csv")
X = data.drop("Churned", axis=1).values
y = data["Churned"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1).to(device)

## Base Model (No Regularization)

In [ ]:
class BaseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

def train_and_plot(model, optimizer, epochs=30, batch_size=32, title="Model"):
    loss_fn = nn.BCELoss()
    train_loss_history, val_loss_history = [], []
    num_batches = len(X_train_tensor) // batch_size
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for i in range(num_batches):
            X_batch = X_train_tensor[i*batch_size:(i+1)*batch_size]
            y_batch = y_train_tensor[i*batch_size:(i+1)*batch_size]
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_loss_history.append(epoch_loss / num_batches)
        model.eval()
        with torch.no_grad():
            val_pred = model(X_test_tensor)
            val_loss = loss_fn(val_pred, y_test_tensor)
            val_loss_history.append(val_loss.item())
        if (epoch+1) % 10 == 0:
            print(f"{title} Epoch {epoch+1}: Train {epoch_loss/num_batches:.4f}, Val {val_loss.item():.4f}")
    plt.plot(train_loss_history, label="Train")
    plt.plot(val_loss_history, label="Val")
    plt.title(title)
    plt.legend()
    plt.show()
    return train_loss_history, val_loss_history

base_model = BaseModel().to(device)
base_opt = optim.SGD(base_model.parameters(), lr=0.01)
train_and_plot(base_model, base_opt, title="Base Model")

## Dropout Model (p=0.4)

In [ ]:
class DropoutModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

dropout_model = DropoutModel().to(device)
dropout_opt = optim.SGD(dropout_model.parameters(), lr=0.01)
train_and_plot(dropout_model, dropout_opt, title="Dropout Model (p=0.4)")

## Weight Decay (L2) Model

In [ ]:
l2_model = BaseModel().to(device)
l2_opt = optim.SGD(l2_model.parameters(), lr=0.01, weight_decay=1e-4)
train_and_plot(l2_model, l2_opt, title="L2 Weight Decay")